## Intermediate Data Science

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209 -- [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/data201_intermediate.html)
- [Syllabus](https://joannabieri.com/data201/IntermediateDataScience.pdf)

## Today's Reading

*Python for Data Analysis*, Chapter 11 - Time Series. The notes below follow the book fairly closely, so keep a notebook open and try the commands as you go.

## Career Discussion (in class today)

*Build a Career in Data Science*, 2.1 Data Science Companies: Massive Tech. We talk about it at the start of class, so bring your notes.

## Time Series Data

Time series are a form of data that are common in many fields: Economics, Finance, Ecology, Neuroscience, Physics, and Applied Math. Any data that is recorded at many points over time, with a certain frequency of observations, is time series data. Time plays an important role in the data - maybe you want to know how observations change over time. 

- *Fixed Frequency* observations are recorded at a fixed interval. The intervals are regular and time steps consistent.
- *Irregular* observations do not have a fixed interval, although time is recorded and could be important to your analysis.

There are many ways that you might mark time series data:

- **Timestamps** time marked by recording specific instants in time.
- **Fixed Periods** time marked by recording the month or the year.
- **Intervals of Time** mark both a start and end timestamp.
- **Experimental or Elapsed Time** time as recorded relative to a start time.

We will mostly explore timestamps, since the other types can usually be converted into timestamps.

In [ ]:
# Some basic package imports
import os
import numpy as np
import pandas as pd

# Visualization packages
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = 'notebook_connected'
import seaborn as sns

## Date and Time Tools

We will start by exploring tools that are available for interacting with dates and times. We will look at the datetime package. 

In [ ]:
from datetime import datetime

In [ ]:
# Lets look at how time is recorded
now = datetime.now()
print(now)

We see the format is year, month, day, hour, minute, second, microsecond. We can access items from this datetime object. Try now. and press the tab button!

In [ ]:
now.year

In [ ]:
now.day

In [ ]:
now.month

Once you have a datetime object you can do operations like addition and subtraction.

**How long have you been alive?**

In [ ]:
my_birthday = datetime(1979,2,7)
my_birthday

In [ ]:
my_life = now-my_birthday
my_life

## You Try

Write a python **function** that takes as an input a birth date and outputs how old the person is in just years, by using the datetime functionality.

In [ ]:
# Your code here


You can add or subtract a `timedelta` to shift a datetime object or create a series of datetimes.

In [ ]:
from datetime import timedelta
# timedelta(days=0, seconds=0, microseconds=0, milliseconds=0, minutes=0, hours=0, weeks=0)

start = datetime(2025,9,3,1,15)
print(start)
delta = timedelta(hours=1,minutes=5)
print(delta)
print(start+delta)

In [ ]:
# You can quickly generate lists of time data
delta = timedelta(days=1)
days_of_year = [datetime(2025,1,1)]

for i in range(30):
    days_of_year.append(days_of_year[-1]+delta)

In [ ]:
days_of_year

## Converting Strings

Often when you read in data from a .csv the data will be in the string format. You can go back and forth between datetime objects and strings.

In [ ]:
dt = datetime(2000,1,1)

# The str() command can convert data into strings
str(dt)

In [ ]:
# You can also format the strings
'''
%Y year
%y two digit year
%m month
%d day

There are lots of these! - see book page 359
'''

dt.strftime("%m-%d-%Y")

In [ ]:
# Start with a string
str_date = '1-1-2000'

# You have to use the string formatting commands to tell date time how the string is formatted
# so it can strip out the parts correctly
datetime.strptime(str_date, "%m-%d-%Y")

## Time Series Basics

We will load in some weather data so we can explore how to deal with time series data. We will set the index to be time series data so that we can explore how we might "slice" or subset time.

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("parthdande/timeseries-weather-dataset")

print("Path to dataset files:", path)

print(os.listdir(path))

In [ ]:
df = pd.read_csv(path+'/'+'Weather_Data_1980_2024(hourly).csv')
df.head(5)

In [ ]:
# Notice that the objects in the time column are strings
df['time'].iloc[0]

In [ ]:
# Change the times to datetime objects
# Save them as the index for the data frame.
def string_to_time(x):
    '''
    A function to return a datetime stripped from the format:
    
    '1980-01-01T00:00'
    
    '''
    return datetime.strptime(x,'%Y-%m-%dT%H:%M')


# Set the index using the time column and a lambda    
df.index = df['time'].apply(lambda x: string_to_time(x))
# Convert the time to datetime
df['time'] = df['time'].apply(lambda x: string_to_time(x))
df

In [ ]:
# Let's look at the data again
df['time'].iloc[0]

Notice that now we have a Timestamp object. Timestamps are a Pandas/Numpy object, basically what happens is when you send a datetime into pandas it interprets it as a Timestamp. We saw the other day that we can call .year, .month, etc on a Timestamp just like above.

Note: pandas.Timestamp stores extra data: nanosecond level precision and frequency information. So it is always safe to convert from datetime to Timestamp, but you might lose some information if you go the other direction.

## Indexing, Selection, and Subsetting

How do you select data that is in Timestamp format? 

In [ ]:
# If we know the exact time stamp we can use it
df.loc['1980-01-01 00:00:00']

In [ ]:
# Pandas will interpret our results if we leave out hours and minutes
df.loc['1980-01-01']

In [ ]:
# It will also interpret strings, even if they are not in the exact order
df.loc['01-01-1980']

In [ ]:
# If you put something in that it can't interpret you get
# OutOfBoundsDatetime: Out of bounds nanosecond timestamp: 1-01-01 00:00:00

# This would give an error
# df.loc['01']

In [ ]:
# You can slice the data by sending in date ranges
# Since time data is chronological you can even use dates not in the range
df.loc['1908':'1981']

In [ ]:
# You can do the same thing with masks
mask = (df['time'] > '1980') & (df['time'] < '1982')
df[mask]

In [ ]:
# You can check for duplicate time stamps just like normal
df['time'].is_unique

In [ ]:
df['time'].duplicated()

## Date Ranges, Frequencies, and Shifting

Often when dealing with dates you need to do some work to make them regular relative to a fixed frequency, even if that means introducing missing variables into your data set. 

You can check the frequency using the pandas function `pd.infer_freq` to see what pandas thinks is the frequency between observations:

In [ ]:
# For our weather data it is recorded hourly
pd.infer_freq(df.index)

If this command outputs nothing, this means it could not infer a frequency from the given data.

### Generating Date Ranges

If you have some data that you know has a particular date range, you can generate date range data using pandas without typing in the dates individually. To do this you need to choose a frequency. Here is a list of common ones:

| Alias          | Description                        |
|----------------|------------------------------------|
| `B`            | Business day frequency             |
| `D`            | Calendar day                       |
| `W` / `W-MON`  | Weekly (optionally anchored)       |
| `ME`           | Month end                          |
| `SME`          | Semi-month end                     |
| `BME`          | Business month end                 |
| `CBME`         | Custom business month end          |
| `MS`           | Month start                        |
| `SMS`          | Semi-month start                   |
| `BMS`          | Business month start               |
| `CBMS`         | Custom business month start        |
| `QE`           | Quarter end                        |
| `QE-JAN`       | Quarter end (ending in January)    |
| `QS`           | Quarter start                      |
| `BQS`          | Business quarter start             |
| `YE`           | Year end                           |
| `YE-APR` etc.  | Fiscal year end (anchored)         |
| `YS`           | Year start                         |
| `h`            | Hourly                             |
| `bh`           | Business hour                      |
| `cbh`          | Custom business hour               |
| `min`          | Minutely                           |
| `s`            | Secondly                           |
| `ms`           | Millisecond                        |
| `us`           | Microsecond                        |
| `ns`           | Nanosecond                         |


In [ ]:
# By default the frequency is Day
dates = pd.date_range('1-1-2025','1-1-2026')
dates

In [ ]:
# Months
dates = pd.date_range('1-1-2025','1-1-2026',freq='ME')
dates

In [ ]:
# Quarters
dates = pd.date_range('1-1-2025','1-1-2026',freq='QE-JAN')
dates

In [ ]:
# Generate a certain number starting at a date
dates = pd.date_range(start='1-1-2025',periods=10, freq='D')
dates

## Frequencies and Offsets

You can use fancier frequencies to get more refined offsets for your dates

In [ ]:
# Here 4h is every 4 hours
dates = pd.date_range(start='1-1-2025',periods=10, freq='4h')
dates

In [ ]:
# Here we use the WOM = week of month to get every 3rd Friday
dates = pd.date_range(start='1-1-2025',periods=10, freq='WOM-3FRI')
dates

For more complicated date ranges you would need to write your own frequency functions

In [ ]:
# Generate daily dates
weeks = 14
all_days = pd.date_range(start='9-1-2026',periods=7*weeks, freq='d')

# Filter for Tuesday (1) and Thursday (3)
tth_days = all_days[all_days.weekday.isin([1, 3])]
print(tth_days)

## Shifting Date Data

Sometimes you want to move data backward or forward in time. Pandas has a `.shift` method for doing this.

In [ ]:
tth_days.shift(1,freq='d')

## Time Zones

One of the hardest things to deal with in time series data is often time zones. Users who enter data using different time zones can really confuse the ordering of a data set. We typically will reference time zones with respect to UTC or coordinated universal time. Then time zones are referenced from the UTC, so for example Redlands is UTC - 7 during Daylight Saving Time (PDT) and UTC - 8 during Standard Time (PST).

Another thing to beware of is that historically, the UTC offsets and things like Daylight Savings have been changed. So be very careful when comparing times across historical data. If you run into issues with time zones for your data you should explore the pytz package. This package has access to a database that contains world time zone information. 

The book has a chapter on dealing with Time Zone data starting on page 374. I am going to skip it here so we don't get too into the weeds!

## Periods and Period Arithmetic

A `Period` in pandas represents a **span of time** (e.g., a day, a month, a quarter), not just a single timestamp. It is useful for period-based time series data where the concept of a **time interval** is more relevant than an exact point in time.

```python
import pandas as pd

pd.Period('2025-09', freq='M')   # Represents September 2025
pd.Period('2025Q3', freq='Q')    # Represents Q3 of 2025
pd.Period('2025-09-30', freq='D')  # Represents the full day of Sept 30, 2025
```

### Why would we use Periods?

1. Time Logic - Grouping

`Period` makes it easy to **group and summarize** time series data by months, quarters, etc.

This avoids confusion from grouping by exact timestamps and ensures consistent aggregation.

2. Avoids Timestamp Precision Errors

Timestamps are overly precise (down to nanoseconds), which may be unnecessary or even problematic for grouped data like "September 2025". `Period` avoids that overprecision.

3. Time Arithmetic at the Period Level

You can do intuitive arithmetic with `Period`:


In [ ]:
# We can define a period as an object itself
p = pd.Period('2025-09', freq='M')
p

In [ ]:
# Then add or subtract
# The amount is based on the freq given
p + 1

In [ ]:
# We can look at the edges of the periods
print(p.start_time)
print(p.end_time)

In [ ]:
# We can use the period to group our data
# Let's find the average monthly temperature in our weather data
df['period'] = df['time'].dt.to_period('M')
df['temperature'].groupby(by=df['period']).mean()


| Feature         | `Timestamp`                      | `Period`                         |
|----------------|----------------------------------|----------------------------------|
| Represents      | A specific moment in time        | A span of time (with frequency)  |
| Useful for      | High-frequency or exact time ops | Aggregated or period-based data |
| Example         | `'2025-09-30 14:00'`             | `'2025-09'` with freq `'M'`      |
| Time arithmetic | Continuous time                  | Discrete period steps            |


Use `Period` when:
- You're working with **monthly, quarterly, or yearly summaries**
- You want **clarity** around time intervals
- You want to **group data cleanly** without dealing with timestamp overprecision


### Quarterly Data

Financial data is often reported quarterly or relative to a fiscal year end. Using periods can help us get dates depending on the quarter and fiscal year. Here is a quick example:

In [ ]:
p = pd.Period('2025Q4', freq='Q-JAN')
p

In [ ]:
# Look to see the start and ends dates of this quarter
print(p.asfreq('D', how='start'))
print(p.asfreq('D', how='end'))

In [ ]:
# What about next quarter?
p = p+1
print(p.asfreq('D', how='start'))
print(p.asfreq('D', how='end'))

You can use the methods `.to_timestamp()` and `.to_period` to convert back and forth between Timestamp data and Period Data. 

In [ ]:
p.asfreq('D', how='start').to_timestamp()

In [ ]:
df['time'].iloc[0].to_period(freq='h')

## You try

Use the methods from lecture Timestamp, Period, and groupby() to find the maximum and minimum temperatures for each year in our weather data. Plot both the max and min temperatures together on a line graph with the years on the x-axis.

In [ ]:
# Your code here

In [ ]:
# Your plot here

## Resampling and Frequency Conversion

But what if you had data that was missing some measurements or data that contained too many measurements? In these cases you want to use resampling. Resampling is the process of changing the frequency of your time series data. It lets you:

- Downsample: Convert high-frequency data (e.g., minute-level) to lower frequency (e.g., daily), usually by aggregating.
- Upsample: Convert lower-frequency data (e.g., daily) to higher frequency (e.g., hourly), often by filling or interpolating values.


Pandas has the `.resample` method to help us with this process. It is similar to `.groupby()` in that it requires a way to aggregate the data before you get back a data frame.

NOTE - your data frame must have a datetime-like index such as:

- DatetimeIndex
- PeriodIndex
- TimedeltaIndex

for resample to work. It always uses the index values

### Downsampling

Downsampling is converting from higher frequency to lower frequency.

In [ ]:
# Let's downsample to get data only yearly
# I will look at just a small number of columns
cols = ['temperature', 'precipitation (mm)','pressure_msl (hPa)']
sample = df[cols].resample('YE')
sample

At this point pandas is ready to return the information but needs to know how to combine the groups. In this case let's return the average values.

In [ ]:
sample.mean()

In [ ]:
# We could also downsample to every 2 hours
sample = df[cols].resample('2h')
sample

In [ ]:
sample.max()

There are lots of ways to play with the data using sampling!!

**Open-high-low-close** resampling

In finance, often we want to compute four important values:

| Term      | Meaning                          |
| --------- | -------------------------------- |
| **Open**  | First price in the time window   |
| **High**  | Highest price in the time window |
| **Low**   | Lowest price in the time window  |
| **Close** | Last price in the time window    |

Pandas has a function for this called `.ohlc()`. Let's see what this does with our temperature data on a daily frequency.


In [ ]:
sample = df['temperature'].resample('D')
sample.ohlc()

### Upsampling

When we did a downsample we had to aggregate the data so that many rows are grouped into one. Upsampling is converting from lower frequency to higher frequency. When we upsample we have to add new rows and decide how we might fill them in.


| Method        | Code Example                     | Use Case                                |
| ------------- | -------------------------------- | --------------------------------------- |
| Forward fill  | `df.resample('h').ffill()`       | Stock prices, step functions            |
| Backward fill | `df.resample('h').bfill()`       | Data where future value applies earlier |
| Interpolate   | `df.resample('h').interpolate()` | Continuous numeric data                 |
| As-is (NaN)   | `df.resample('h').asfreq()`      | When you want to leave gaps             |

Let's start with our weather data, but pretend like we only know the values weekly:

In [ ]:
df_example = df[cols].resample('W').mean()
df_example

Here we are pretending that we don't know the original data - these are our only observations! Now what if we wanted to expand this data to daily observations?

The first cell will look the same as everything above! We just changed our sample from WEEK to DAY

In [ ]:
sample = df_example.resample('D')
sample

In [ ]:
# Now we need to aggregate - or interpolate
sample.asfreq().head(15)

The aggregation `.asfreq()` converts to the higher frequency without any aggregation and just inserts NaN where it does not have data. Lets try some of the other methods!

In [ ]:
sample.ffill().head(15)

In [ ]:
sample.interpolate().head(15)

## Moving Window Functions

Next we will consider functions that are evaluated over a sliding window of time or evaluated with exponentially decaying weights. Our book calls these "moving window functions"


When analyzing time series data, we often want to extract meaningful trends without being overwhelmed by short-term noise. Two powerful techniques for this are **sliding window functions** and **exponentially weighted functions**.


### Sliding Window Functions (Rolling Windows)

These compute metrics (like `mean`, `sum`, `std`, etc.) over a fixed-size window that "slides" across the time series.

**Use cases:**
- 7-day moving average of temperature
- 30-day rolling volatility of returns
- Smoothing daily sales data

**Why it's useful:**
- Helps observe short-term trends over time
- Reduces the influence of sudden spikes or dips

For this we will use the `.rolling()` method. Instead of looking at the weather data here we will read in the stock data from before, this data is more illustrative of the method and follows the book.



In [ ]:
file = 'data/stock_px.csv'

# NOW we can talk about what the extra commands do here!
# parse_dates = True, tells pandas to turn dates into Timestamps
# index_col = 0. sends the first column to be the index
# in this data that means the timestamp is the index and we need that for our methods!
df_stocks = pd.read_csv(file,parse_dates=True,index_col=0)
df_stocks

Now we will resample. This data looks to be daily frequency, but maybe we actually want to have the information based on the business day frequency.

| Frequency          | Code  | Includes                                    |
| ------------------ | ----- | ------------------------------------------- |
| **Daily**          | `'D'` | All calendar days (Mon–Sun)                 |
| **Business Daily** | `'B'` | Only weekdays (Mon–Fri) — excludes weekends |


In [ ]:
# Resample with a forward fill - some days are missing and we remove weekends.
df_resample = df_stocks.resample('B').ffill()
df_resample

In [ ]:
# Plot the data for AAPL
my_col = 'AAPL'


plt.plot(df_resample.index,df_resample[my_col],'r-',linewidth=.5)
plt.grid()
plt.xlabel('Day')
plt.ylabel('Price')

plt.show()

We notice how financial data has lots of ups and downs and if we zoom into the data the change from one day to the next actually tells us very little. This is why we often use rolling functions to understand the data. Here we will calculate a rolling average and plot it with the data.

Here we will calculate a mean over a 250 day window. You have to choose what window to use! As the window slides across the timeseries the data on the right becomes part of the average and the data from the left leaves the average.

In [ ]:
rolling_ave = df_resample[my_col].rolling(250).mean()

plt.plot(df_resample.index,df_resample[my_col],'r-',linewidth=.5,label='Data')
plt.plot(df_resample.index,rolling_ave,'k-',linewidth=.8,label='Rolling Average')
plt.grid()
plt.xlabel('Day')
plt.ylabel('Price')
plt.legend()

plt.show()

In [ ]:
# Do a rolling average of 100 days
rolling_ave = df_resample[my_col].rolling('100D').mean()

plt.plot(df_resample.index,df_resample[my_col],'r-',linewidth=.5,label='Data')
plt.plot(df_resample.index,rolling_ave,'k-',linewidth=.8,label='Rolling Average')
plt.grid()
plt.xlabel('Day')
plt.ylabel('Price')
plt.legend()

plt.show()

By default `.rolling()` cannot deal with NaN values. However there is an optional flag `min_periods=` which lets you specify the minimum number of non-nan values that can be used to calculate. This way NaNs can be dropped.

Sometimes instead of a rolling window, you want an expanding window. For example we could calculate the average as we expand our data over time. In this case the right edge of the window expands to include more data in the average.

In [ ]:
expand_ave = df_resample[my_col].expanding().mean()

plt.plot(df_resample.index,df_resample[my_col],'r-',linewidth=.5,label='Data')
plt.plot(df_resample.index,expand_ave,'k-',linewidth=.8,label='Expanding Average')
plt.grid()
plt.xlabel('Day')
plt.ylabel('Price')
plt.legend()

plt.show()

### Exponentially Weighted Functions (EWM)

These compute statistics using exponentially decaying weights, giving more importance to more recent data points.

**Use cases:**
- Real-time trend tracking (e.g., financial indicators)
- Adaptive smoothing for changing behavior
- Faster reaction to recent changes compared to rolling averages

**Why it's useful:**
- More responsive to recent data
- Does not require a fixed window size
- Better suited for evolving or rapidly changing data

In Pandas we will use the `ewm()` exponentially weighted moving function. Here we think of applying a decay factor based on the span which determines how much memory the ewm has. Often we combine exponential weighting with moving average so that the most recent data has more of an impact on the outcome.

In [ ]:
rolling_ave = df_resample[my_col].rolling(250).mean()

# We apply the exponential weighting to the rolling data
# Here we use a span of 30 days.
ewm_rolling_ave = rolling_ave.ewm(span=30).mean()

plt.plot(df_resample.index,df_resample[my_col],'r-',linewidth=.5,label='Data')
plt.plot(df_resample.index,rolling_ave,'k-',linewidth=.8,label='Rolling Average')
plt.plot(df_resample.index,ewm_rolling_ave,'b-',linewidth=.8,label='EWM Rolling Average')
plt.grid()
plt.xlabel('Day')
plt.ylabel('Price')
plt.legend()

plt.show()

### Binary Moving Window Functions

When you are calculating things that need more than one set of timeseries data, for example correlation or covariance, you need to send in additional data into the functions. Here we will plot the correlation between the percent change in the stock price of 'AAPL' compared to the percent change in the benchmark index 'SPX'

In [ ]:
# Get the percent changes
pcng_spx = df_resample['SPX'].pct_change()
pcng_aapl = df_resample['AAPL'].pct_change()

In [ ]:
# Now calculate the correlation
corr = pcng_aapl.rolling(250).corr(pcng_spx)

In [ ]:
plt.plot(df_resample.index,corr,'r-',linewidth=.8,label='Correlation Coef')
plt.grid()
plt.xlabel('Day')
plt.ylabel('Correlation')
plt.legend()
plt.ylim([0,1])

plt.show()

## You Try

Add the other two stocks to the correlation graph above. Make sure to label the graph clearly and make it look nice! Experiment with different values for your moving average window. What happens when you change this and why?

In [ ]:
# Your code here

## Homework 9

The full assignment is in `HW_day9.ipynb` in your sandbox. The short version: German electricity data (consumption, wind, and solar production), turn the dates into a time series, and use resampling and rolling windows to find the patterns over days, weeks, and years.

Work the problems in your sandbox. Your team's write-up notebook goes in `Week05` of your team repo. Homework 8 and Homework 9 both go in `Week05` and are due Sunday 10/4 at 11:59pm.

Exam 1 is handed out on Tuesday 10/6.